# U.S. County Economic Stress & Food Insecurity Data Pipeline

## Purpose

Build a reproducible county-level master dataset for the U.S. County Economic
Stress & Food Insecurity Explorer.

The dataset integrates economic, housing, food insecurity, health insurance,
and geographic indicators for U.S. counties and county equivalents.

## Data Sources

- Economic Stress U.S. Counties Master
  - U.S. Census Bureau SAIPE, 2023
  - U.S. Census Bureau ACS 5-Year Estimates, 2023
  - U.S. Bureau of Labor Statistics LAUS, 2025
- Feeding America, Map the Meal Gap 2026 (2024 data)
- U.S. Census Bureau SAHIE, 2023

In [5]:
from pathlib import Path
import pandas as pd

Path.cwd()

PosixPath('/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/notebooks')

In [6]:
# Project paths
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"

MASTER_PATH = RAW_DIR / "Economic_Stress_US_Counties_Master.xlsx"
FEEDING_AMERICA_PATH = RAW_DIR / "MMG2026_2024_Data_To_Share.xlsx"

print("Project:", PROJECT_DIR)
print("Master exists:", MASTER_PATH.exists())
print("Feeding America exists:", FEEDING_AMERICA_PATH.exists())

Project: /Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer
Master exists: True
Feeding America exists: True


In [7]:
# Load existing Economic Stress county master
master = pd.read_excel(
    MASTER_PATH,
    sheet_name="Sheet1",
    dtype={
        "State_FIPS": str,
        "County_FIPS": str
    }
)

print("Shape:", master.shape)
master.head()

Shape: (3144, 14)


,State_FIPS,County_FIPS,State,County,Poverty_All_Pct,Poverty_Index,Severe_Burden_Pct,Housing_Index,Unemployment_Pct,Unemployment_Index,Economic_Stress_Score,Median_Household_Income,High_Burden_35Plus_Pct,Renter_Pop
0,01,001,AL,Autauga County,12.9,0.175,15.0770,0.236,2.4,0.093,16.8,78840.0,24.08423,Large
1,01,003,AL,Baldwin County,10.5,0.129,18.3905,0.287,2.5,0.099,17.2,82519.0,34.16584,Large
2,01,005,AL,Barbour County,28.1,0.468,14.3970,0.225,3.6,0.159,28.4,45902.0,24.86450,Large
3,01,007,AL,Bibb County,21.6,0.343,22.3380,0.349,2.7,0.110,26.7,60359.0,36.63194,Large
4,01,009,AL,Blount County,11.7,0.152,17.9396,0.280,2.4,0.093,17.5,69371.0,33.65897,Large


In [8]:
pd.ExcelFile(MASTER_PATH).sheet_names

['ReadMe', 'Sheet1']

In [9]:
# Baseline QA for existing county master

print(f"Rows: {len(master):,}")
print(f"Columns: {master.shape[1]}")
print(f"Unique State FIPS: {master['State_FIPS'].nunique()}")
print(f"Unique county records: {master[['State_FIPS', 'County_FIPS']].drop_duplicates().shape[0]:,}")

print("\nMissing values by column:")
display(master.isna().sum())

print("\nDuplicate county keys:")
duplicates = master.duplicated(
    subset=["State_FIPS", "County_FIPS"],
    keep=False
)
print(duplicates.sum())

Rows: 3,144
Columns: 14
Unique State FIPS: 51
Unique county records: 3,144

Missing values by column:


State_FIPS                 0
County_FIPS                0
State                      0
County                     0
Poverty_All_Pct            1
Poverty_Index              1
Severe_Burden_Pct          1
Housing_Index              1
Unemployment_Pct           1
Unemployment_Index         1
Economic_Stress_Score      1
Median_Household_Income    1
High_Burden_35Plus_Pct     0
Renter_Pop                 0
dtype: int64


Duplicate county keys:
0


In [10]:
# Standardize FIPS components and create 5-digit county FIPS

master["State_FIPS"] = master["State_FIPS"].str.zfill(2)
master["County_FIPS"] = master["County_FIPS"].str.zfill(3)

master["FIPS"] = master["State_FIPS"] + master["County_FIPS"]

master[["State", "County", "State_FIPS", "County_FIPS", "FIPS"]].head(10)

,State,County,State_FIPS,County_FIPS,FIPS
0,AL,Autauga County,01,001,01001
1,AL,Baldwin County,01,003,01003
2,AL,Barbour County,01,005,01005
3,AL,Bibb County,01,007,01007
4,AL,Blount County,01,009,01009
5,AL,Bullock County,01,011,01011
6,AL,Butler County,01,013,01013
7,AL,Calhoun County,01,015,01015
8,AL,Chambers County,01,017,01017
9,AL,Cherokee County,01,019,01019


In [11]:
print("Unique FIPS:", master["FIPS"].nunique())
print("All FIPS are 5 digits:", master["FIPS"].str.fullmatch(r"\d{5}").all())

Unique FIPS: 3144
All FIPS are 5 digits: True


In [12]:
pd.ExcelFile(FEEDING_AMERICA_PATH).sheet_names

['Read Me',
 'County',
 'Congressional District',
 'State',
 'Metro Area (50+ Only)',
 'National (50+ Only)']

In [13]:
feeding_america = pd.read_excel(
    FEEDING_AMERICA_PATH,
    sheet_name="County",
    dtype={"FIPS": str}
)

print("Shape:", feeding_america.shape)
feeding_america.head()

Shape: (3144, 25)


,FIPS,State,"County, State",Year,Overall Food Insecurity Rate,# of Food Insecure Persons Overall,Food Insecurity Rate among Black Persons (all ethnicities),Food Insecurity Rate among Hispanic Persons (any race),"Food Insecurity Rate among White, non-Hispanic Persons",SNAP Threshold,...,% food insecure children in HH w/ HH incomes above 185 FPL,Imputed Market Basket Data,Cost Per Meal,Weighted weekly $ needed by FI,Weighted Annual Food Budget Shortfall,Rural-Urban Continuum Code (2013),Rural-Urban Continuum Code (2023),Census Region,Census Division,FNS Region
0,1001,AL,"Autauga County, Alabama",2024,0.156,9350.0,0.26,NaN,0.12,1.3,...,0.33,NaN,3.68,23.02,6528000.0,2.0,2,South,East South Central,Southeast
1,1003,AL,"Baldwin County, Alabama",2024,0.154,37940.0,0.30,0.18,0.11,1.3,...,0.36,NaN,4.15,25.95,29861000.0,3.0,3,South,East South Central,Southeast
2,1005,AL,"Barbour County, Alabama",2024,0.193,4750.0,0.33,0.25,0.14,1.3,...,0.06,NaN,3.53,22.08,3181000.0,6.0,6,South,East South Central,Southeast
3,1007,AL,"Bibb County, Alabama",2024,0.202,4470.0,0.47,NaN,0.15,1.3,...,0.15,NaN,3.43,21.49,2914000.0,1.0,1,South,East South Central,Southeast
4,1009,AL,"Blount County, Alabama",2024,0.184,10960.0,0.28,0.22,0.13,1.3,...,0.29,NaN,3.44,21.56,7167000.0,1.0,1,South,East South Central,Southeast


In [14]:
feeding_america["FIPS"] = feeding_america["FIPS"].str.zfill(5)

print("Unique FIPS:", feeding_america["FIPS"].nunique())
print("Duplicate FIPS:", feeding_america["FIPS"].duplicated().sum())
print("All FIPS are 5 digits:", feeding_america["FIPS"].str.fullmatch(r"\d{5}").all())

Unique FIPS: 3144
Duplicate FIPS: 0
All FIPS are 5 digits: True


In [15]:
feeding_america.columns.tolist()

['FIPS',
 'State',
 'County, State',
 'Year',
 'Overall Food Insecurity Rate',
 '# of Food Insecure Persons Overall',
 'Food Insecurity Rate among Black Persons (all ethnicities)',
 'Food Insecurity Rate among Hispanic Persons (any race)',
 'Food Insecurity Rate among White, non-Hispanic Persons',
 'SNAP Threshold',
 '% FI ≤ SNAP Threshold',
 '% FI > SNAP Threshold',
 'Child Food Insecurity Rate',
 '# of Food Insecure Children',
 '% food insecure children in HH w/ HH incomes below 185 FPL',
 '% food insecure children in HH w/ HH incomes above 185 FPL',
 'Imputed Market Basket Data',
 'Cost Per Meal',
 'Weighted weekly $ needed by FI',
 'Weighted Annual Food Budget Shortfall',
 'Rural-Urban Continuum Code (2013)',
 'Rural-Urban Continuum Code (2023)',
 'Census Region',
 'Census Division',
 'FNS Region']

In [16]:
feeding_america.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3144 entries, 0 to 3143
Data columns (total 25 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   FIPS                                                        3144 non-null   object 
 1   State                                                       3144 non-null   object 
 2   County, State                                               3144 non-null   object 
 3   Year                                                        3144 non-null   int64  
 4   Overall Food Insecurity Rate                                3142 non-null   float64
 5   # of Food Insecure Persons Overall                          3142 non-null   float64
 6   Food Insecurity Rate among Black Persons (all ethnicities)  1478 non-null   float64
 7   Food Insecurity Rate among Hispanic Persons (any race)      1860 non-null   float64
 8 

In [17]:
feeding_america.head(3).T

,0,1,2
FIPS,01001,01003,01005
State,AL,AL,AL
"County, State","Autauga County, Alabama","Baldwin County, Alabama","Barbour County, Alabama"
Year,2024,2024,2024
Overall Food Insecurity Rate,0.156,0.154,0.193
# of Food Insecure Persons Overall,9350.0,37940.0,4750.0
Food Insecurity Rate among Black Persons (all ethnicities),0.26,0.3,0.33
Food Insecurity Rate among Hispanic Persons (any race),NaN,0.18,0.25
"Food Insecurity Rate among White, non-Hispanic Persons",0.12,0.11,0.14
SNAP Threshold,1.3,1.3,1.3


In [18]:
readme = pd.read_excel(
    FEEDING_AMERICA_PATH,
    sheet_name="Read Me",
    header=None
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(readme)

,0,1,2,3,4
0,Dataset that combines Feeding America's Map the Meal Gap 2026: An Analysis of County and Congressional District Food Insecurity and County Food Cost in the United States in 2024,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,Recommended Citation,NaN,NaN,NaN,NaN
3,"Ribar, D.C., Harris, V., Dewey, A., Dawes, S., Hilvers, J., and Engelhard, E. (2026). Map the Meal Gap: An Analysis of Local Food Insecurity and Food Costs in the United States in 2024. Feeding America National Organization.",NaN,NaN,NaN,NaN
4,Version History,Note,NaN,NaN,NaN
5,MMG2026_2024_Data_To_Share,"This file was last updated on July 28, 2026.",NaN,NaN,NaN
6,General Notes,NaN,NaN,NaN,NaN
7,"All overall, child, race/ethnicty food insecurity estimates produced by Feeding America as part of the Map the Meal Gap project are modeled estimates and should be interpreted as approximations (e.g., a community with an estimated food insecurity rate of 0% may still contain individuals and/or households that experience food insecurity).",NaN,NaN,NaN,NaN
8,"Blank cells indicate that values are currently unavailable, not applicable and/or data required to produce local estimates were unavailable.",NaN,NaN,NaN,NaN
9,Notes on Revised 2023 Estimates,NaN,NaN,NaN,NaN


In [19]:
feeding_america.loc[
    feeding_america["Overall Food Insecurity Rate"].isna(),
    ["FIPS", "State", "County, State", "Year"]
]

,FIPS,State,"County, State",Year
550,15005,HI,"Kalawao County, Hawaii",2024
2675,48301,TX,"Loving County, Texas",2024


### Missing County Food Insecurity Estimates

Feeding America's 2024 county dataset contains records for all 3,144
U.S. counties and county equivalents. Overall and child food insecurity
estimates are available for 3,142.

The two counties without reported food insecurity estimates are:

- Kalawao County, Hawaii (FIPS 15005)
- Loving County, Texas (FIPS 48301)

According to Feeding America's documentation, blank cells indicate that
values are unavailable, not applicable, and/or that data required to
produce local estimates were unavailable.

These counties will remain in the master dataset, with unavailable
Feeding America measures retained as missing values rather than being
imputed or interpreted as zero.

In [20]:
# Select and rename Feeding America fields for the canonical master

fa_selected = feeding_america[
    [
        "FIPS",
        "Year",
        "Overall Food Insecurity Rate",
        "# of Food Insecure Persons Overall",
        "SNAP Threshold",
        "% FI ≤ SNAP Threshold",
        "% FI > SNAP Threshold",
        "Child Food Insecurity Rate",
        "# of Food Insecure Children",
        "Cost Per Meal",
        "Weighted Annual Food Budget Shortfall",
        "Rural-Urban Continuum Code (2023)",
        "Census Region",
        "Census Division",
    ]
].copy()

fa_selected = fa_selected.rename(
    columns={
        "Year": "Food_Insecurity_Year",
        "Overall Food Insecurity Rate": "Food_Insecurity_Rate",
        "# of Food Insecure Persons Overall": "Food_Insecure_Persons",
        "SNAP Threshold": "SNAP_Threshold",
        "% FI ≤ SNAP Threshold": "FI_At_or_Below_SNAP_Threshold_Pct",
        "% FI > SNAP Threshold": "FI_Above_SNAP_Threshold_Pct",
        "Child Food Insecurity Rate": "Child_Food_Insecurity_Rate",
        "# of Food Insecure Children": "Food_Insecure_Children",
        "Cost Per Meal": "Cost_Per_Meal",
        "Weighted Annual Food Budget Shortfall": "Annual_Food_Budget_Shortfall",
        "Rural-Urban Continuum Code (2023)": "RUCC_2023",
        "Census Region": "Region",
        "Census Division": "Census_Division",
    }
)

fa_selected.head()

,FIPS,Food_Insecurity_Year,Food_Insecurity_Rate,Food_Insecure_Persons,SNAP_Threshold,FI_At_or_Below_SNAP_Threshold_Pct,FI_Above_SNAP_Threshold_Pct,Child_Food_Insecurity_Rate,Food_Insecure_Children,Cost_Per_Meal,Annual_Food_Budget_Shortfall,RUCC_2023,Region,Census_Division
0,01001,2024,0.156,9350.0,1.3,0.444,0.556,0.203,2850.0,3.68,6528000.0,2,South,East South Central
1,01003,2024,0.154,37940.0,1.3,0.455,0.545,0.196,10210.0,4.15,29861000.0,3,South,East South Central
2,01005,2024,0.193,4750.0,1.3,0.419,0.581,0.308,1580.0,3.53,3181000.0,6,South,East South Central
3,01007,2024,0.202,4470.0,1.3,0.600,0.400,0.279,1240.0,3.43,2914000.0,1,South,East South Central
4,01009,2024,0.184,10960.0,1.3,0.590,0.410,0.213,2930.0,3.44,7167000.0,1,South,East South Central


In [21]:
# Derive broad metropolitan status from USDA ERS RUCC 2023
# RUCC 1-3 = Metropolitan; RUCC 4-9 = Nonmetropolitan

fa_selected["Metro_Status"] = pd.NA

fa_selected.loc[
    fa_selected["RUCC_2023"].between(1, 3),
    "Metro_Status"
] = "Metropolitan"

fa_selected.loc[
    fa_selected["RUCC_2023"].between(4, 9),
    "Metro_Status"
] = "Nonmetropolitan"

fa_selected[
    ["FIPS", "RUCC_2023", "Metro_Status", "Region", "Census_Division"]
].head(10)

,FIPS,RUCC_2023,Metro_Status,Region,Census_Division
0,01001,2,Metropolitan,South,East South Central
1,01003,3,Metropolitan,South,East South Central
2,01005,6,Nonmetropolitan,South,East South Central
3,01007,1,Metropolitan,South,East South Central
4,01009,1,Metropolitan,South,East South Central
5,01011,8,Nonmetropolitan,South,East South Central
6,01013,6,Nonmetropolitan,South,East South Central
7,01015,3,Metropolitan,South,East South Central
8,01017,6,Nonmetropolitan,South,East South Central
9,01019,8,Nonmetropolitan,South,East South Central


In [22]:
# Validate selected Feeding America dataset

print(f"Rows: {len(fa_selected):,}")
print(f"Columns: {fa_selected.shape[1]}")
print(f"Unique FIPS: {fa_selected['FIPS'].nunique():,}")
print(f"Duplicate FIPS: {fa_selected['FIPS'].duplicated().sum()}")

print("\nFood insecurity year values:")
print(fa_selected["Food_Insecurity_Year"].value_counts(dropna=False))

print("\nMetro status counts:")
print(fa_selected["Metro_Status"].value_counts(dropna=False))

print("\nRegion counts:")
print(fa_selected["Region"].value_counts(dropna=False))

print("\nMissing values:")
display(fa_selected.isna().sum())

Rows: 3,144
Columns: 15
Unique FIPS: 3,144
Duplicate FIPS: 0

Food insecurity year values:
Food_Insecurity_Year
2024    3144
Name: count, dtype: int64

Metro status counts:
Metro_Status
Nonmetropolitan    1958
Metropolitan       1186
Name: count, dtype: int64

Region counts:
Region
South        1422
Midwest      1055
West          449
Northeast     218
Name: count, dtype: int64

Missing values:


FIPS                                 0
Food_Insecurity_Year                 0
Food_Insecurity_Rate                 2
Food_Insecure_Persons                2
SNAP_Threshold                       0
FI_At_or_Below_SNAP_Threshold_Pct    2
FI_Above_SNAP_Threshold_Pct          2
Child_Food_Insecurity_Rate           2
Food_Insecure_Children               2
Cost_Per_Meal                        0
Annual_Food_Budget_Shortfall         2
RUCC_2023                            0
Region                               0
Census_Division                      0
Metro_Status                         0
dtype: int64

In [23]:
# Merge Feeding America fields onto the Economic Stress master

master_enriched = master.merge(
    fa_selected,
    on="FIPS",
    how="left",
    validate="one_to_one"
)

print("Original master shape:", master.shape)
print("Enriched master shape:", master_enriched.shape)

Original master shape: (3144, 15)
Enriched master shape: (3144, 29)


In [24]:
# Confirm all master counties matched a Feeding America geography record

unmatched_fa = master.loc[
    ~master["FIPS"].isin(fa_selected["FIPS"]),
    ["FIPS", "State", "County"]
]

print(f"Unmatched master counties: {len(unmatched_fa)}")
display(unmatched_fa)

Unmatched master counties: 0


,FIPS,State,County


In [25]:
# Post-merge integrity checks

print(f"Rows: {len(master_enriched):,}")
print(f"Unique FIPS: {master_enriched['FIPS'].nunique():,}")
print(f"Duplicate FIPS: {master_enriched['FIPS'].duplicated().sum()}")

print("\nMissing Food Insecurity Rate:")
display(
    master_enriched.loc[
        master_enriched["Food_Insecurity_Rate"].isna(),
        ["FIPS", "State", "County", "Food_Insecurity_Rate"]
    ]
)

Rows: 3,144
Unique FIPS: 3,144
Duplicate FIPS: 0

Missing Food Insecurity Rate:


,FIPS,State,County,Food_Insecurity_Rate
550,15005,HI,Kalawao County,NaN
2675,48301,TX,Loving County,NaN


## Feeding America Integration — Validation

Feeding America's Map the Meal Gap 2026 county-level data (2024 estimates)
were integrated with the Economic Stress master using 5-digit county FIPS.

### Validation Results

- 3,144 county and county-equivalent records in the Economic Stress master
- 3,144 unique county FIPS in the Feeding America dataset
- 3,144 county records retained after the merge
- 3,144 unique FIPS retained
- 0 duplicate FIPS
- 0 unmatched county geographies

Overall food insecurity estimates are available for 3,142 counties and county
equivalents. Feeding America does not report an overall food insecurity estimate
for:

- Kalawao County, Hawaii (FIPS 15005)
- Loving County, Texas (FIPS 48301)

These counties remain in the master dataset, and unavailable Feeding America
measures are retained as missing values rather than imputed or interpreted as zero.

Feeding America notes that its county food insecurity measures are modeled
estimates and should be interpreted as approximations.

## Health Insurance Coverage — SAHIE 2023

In [26]:
try:
    from dotenv import load_dotenv
    print("python-dotenv is installed.")
except ImportError:
    print("python-dotenv needs to be installed.")

python-dotenv is installed.


In [27]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
from dotenv import load_dotenv

# Load environment variables from the project-level .env file
load_dotenv(PROJECT_DIR / ".env")

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

print("Census API key loaded:", CENSUS_API_KEY is not None)

Census API key loaded: True


In [29]:
import requests

SAHIE_URL = "https://api.census.gov/data/timeseries/healthins/sahie"

params = {
    "get": "NAME,GEOID,STABREV,NUI_PT,PCTUI_PT,YEAR",
    "for": "county:*",
    "in": "state:*",
    "time": "2024",
    "AGECAT": "0",
    "SEXCAT": "0",
    "IPRCAT": "0",
    "key": CENSUS_API_KEY,
}

response = requests.get(SAHIE_URL, params=params, timeout=30)

print("Status code:", response.status_code)

Status code: 200


In [30]:
sahie_json = response.json()

sahie = pd.DataFrame(
    sahie_json[1:],
    columns=sahie_json[0]
)

print("Shape:", sahie.shape)
sahie.head()

Shape: (3144, 12)


,NAME,GEOID,STABREV,NUI_PT,PCTUI_PT,YEAR,time,AGECAT,SEXCAT,IPRCAT,state,county
0,"Autauga County, AL",01001,AL,4094,8.1,2024,2024,0,0,0,01,001
1,"Baldwin County, AL",01003,AL,19236,9.5,2024,2024,0,0,0,01,003
2,"Crawford County, AR",05033,AR,6035,11.9,2024,2024,0,0,0,05,033
3,"Barbour County, AL",01005,AL,1864,11.5,2024,2024,0,0,0,01,005
4,"Bibb County, AL",01007,AL,1841,11.4,2024,2024,0,0,0,01,007


In [31]:
print("Unique GEOID:", sahie["GEOID"].nunique())
print("Duplicate GEOID:", sahie["GEOID"].duplicated().sum())
print("Year values:")
print(sahie["YEAR"].value_counts(dropna=False))

Unique GEOID: 3144
Duplicate GEOID: 0
Year values:
YEAR
2024    3144
Name: count, dtype: int64


In [32]:
# Prepare SAHIE 2024 county uninsured measures

sahie_selected = sahie[
    [
        "GEOID",
        "NAME",
        "STABREV",
        "NUI_PT",
        "PCTUI_PT",
        "YEAR"
    ]
].copy()

sahie_selected = sahie_selected.rename(
    columns={
        "GEOID": "FIPS",
        "NUI_PT": "Uninsured_Persons",
        "PCTUI_PT": "Uninsured_Rate",
        "YEAR": "Uninsured_Year"
    }
)

# Convert numeric fields returned by the API as strings
sahie_selected["Uninsured_Persons"] = pd.to_numeric(
    sahie_selected["Uninsured_Persons"],
    errors="coerce"
)

sahie_selected["Uninsured_Rate"] = pd.to_numeric(
    sahie_selected["Uninsured_Rate"],
    errors="coerce"
)

sahie_selected["Uninsured_Year"] = pd.to_numeric(
    sahie_selected["Uninsured_Year"],
    errors="coerce"
)

sahie_selected.head()

,FIPS,NAME,STABREV,Uninsured_Persons,Uninsured_Rate,Uninsured_Year
0,01001,"Autauga County, AL",AL,4094.0,8.1,2024
1,01003,"Baldwin County, AL",AL,19236.0,9.5,2024
2,05033,"Crawford County, AR",AR,6035.0,11.9,2024
3,01005,"Barbour County, AL",AL,1864.0,11.5,2024
4,01007,"Bibb County, AL",AL,1841.0,11.4,2024


In [33]:
print(f"Rows: {len(sahie_selected):,}")
print(f"Unique FIPS: {sahie_selected['FIPS'].nunique():,}")
print(f"Duplicate FIPS: {sahie_selected['FIPS'].duplicated().sum()}")

print("\nMissing values:")
display(sahie_selected.isna().sum())

print("\nUninsured rate range:")
print(
    sahie_selected["Uninsured_Rate"].min(),
    "to",
    sahie_selected["Uninsured_Rate"].max()
)

Rows: 3,144
Unique FIPS: 3,144
Duplicate FIPS: 0

Missing values:


FIPS                 0
NAME                 0
STABREV              0
Uninsured_Persons    1
Uninsured_Rate       1
Uninsured_Year       0
dtype: int64


Uninsured rate range:
2.4 to 38.5


In [34]:
master_final = master_enriched.merge(
    sahie_selected[
        [
            "FIPS",
            "Uninsured_Persons",
            "Uninsured_Rate",
            "Uninsured_Year"
        ]
    ],
    on="FIPS",
    how="left",
    validate="one_to_one"
)

In [35]:
print(f"Rows: {len(master_final):,}")
print(f"Columns: {master_final.shape[1]}")
print(f"Unique FIPS: {master_final['FIPS'].nunique():,}")
print(f"Duplicate FIPS: {master_final['FIPS'].duplicated().sum()}")

print("\nMissing uninsured values:")
display(
    master_final[
        master_final["Uninsured_Rate"].isna()
    ][["FIPS", "State", "County", "Uninsured_Rate"]]
)

Rows: 3,144
Columns: 32
Unique FIPS: 3,144
Duplicate FIPS: 0

Missing uninsured values:


,FIPS,State,County,Uninsured_Rate
550,15005,HI,Kalawao County,NaN


## SAHIE 2024 Integration — Validation

U.S. Census Bureau Small Area Health Insurance Estimates (SAHIE) 2024
county-level uninsured estimates were integrated with the enriched county
master using 5-digit county FIPS.

### Validation Results

- 3,144 county and county-equivalent records retained
- 3,144 unique FIPS retained
- 0 duplicate FIPS
- 0 unmatched county geographies

The SAHIE uninsured rate is unavailable for:

- Kalawao County, Hawaii (FIPS 15005)

This county remains in the master dataset, with the unavailable uninsured
estimate retained as a missing value rather than imputed or interpreted as zero.

## Final Dataset Quality Assurance

The enriched county master is validated prior to export for use in the
U.S. County Economic Stress & Food Insecurity Explorer.

Checks include dataset structure, geographic integrity, missingness,
numeric ranges, categorical dimensions, source years, and selected
county-level spot checks.

In [36]:
print("FINAL DATASET STRUCTURE")
print("-" * 40)

print(f"Rows: {len(master_final):,}")
print(f"Columns: {master_final.shape[1]}")
print(f"Unique FIPS: {master_final['FIPS'].nunique():,}")
print(f"Duplicate FIPS: {master_final['FIPS'].duplicated().sum()}")

print("\nColumn names:")
for i, col in enumerate(master_final.columns, start=1):
    print(f"{i:>2}. {col}")

FINAL DATASET STRUCTURE
----------------------------------------
Rows: 3,144
Columns: 32
Unique FIPS: 3,144
Duplicate FIPS: 0

Column names:
 1. State_FIPS
 2. County_FIPS
 3. State
 4. County
 5. Poverty_All_Pct
 6. Poverty_Index
 7. Severe_Burden_Pct
 8. Housing_Index
 9. Unemployment_Pct
10. Unemployment_Index
11. Economic_Stress_Score
12. Median_Household_Income
13. High_Burden_35Plus_Pct
14. Renter_Pop
15. FIPS
16. Food_Insecurity_Year
17. Food_Insecurity_Rate
18. Food_Insecure_Persons
19. SNAP_Threshold
20. FI_At_or_Below_SNAP_Threshold_Pct
21. FI_Above_SNAP_Threshold_Pct
22. Child_Food_Insecurity_Rate
23. Food_Insecure_Children
24. Cost_Per_Meal
25. Annual_Food_Budget_Shortfall
26. RUCC_2023
27. Region
28. Census_Division
29. Metro_Status
30. Uninsured_Persons
31. Uninsured_Rate
32. Uninsured_Year


In [37]:
for i, col in enumerate(master_final.columns, start=1):
    print(i, repr(col))

1 'State_FIPS'
2 'County_FIPS'
3 'State'
4 'County'
5 'Poverty_All_Pct'
6 'Poverty_Index'
7 'Severe_Burden_Pct'
8 'Housing_Index'
9 'Unemployment_Pct'
10 'Unemployment_Index'
11 'Economic_Stress_Score'
12 'Median_Household_Income'
13 'High_Burden_35Plus_Pct'
14 'Renter_Pop'
15 'FIPS'
16 'Food_Insecurity_Year'
17 'Food_Insecurity_Rate'
18 'Food_Insecure_Persons'
19 'SNAP_Threshold'
20 'FI_At_or_Below_SNAP_Threshold_Pct'
21 'FI_Above_SNAP_Threshold_Pct'
22 'Child_Food_Insecurity_Rate'
23 'Food_Insecure_Children'
24 'Cost_Per_Meal'
25 'Annual_Food_Budget_Shortfall'
26 'RUCC_2023'
27 'Region'
28 'Census_Division'
29 'Metro_Status'
30 'Uninsured_Persons'
31 'Uninsured_Rate'
32 'Uninsured_Year'


In [38]:
final_column_order = [
    # Geography / identifiers
    "FIPS",
    "State_FIPS",
    "County_FIPS",
    "State",
    "County",

    # Economic conditions
    "Poverty_All_Pct",
    "Poverty_Index",
    "Median_Household_Income",
    "Unemployment_Pct",
    "Unemployment_Index",
    "Severe_Burden_Pct",
    "High_Burden_35Plus_Pct",
    "Housing_Index",
    "Renter_Pop",
    "Economic_Stress_Score",

    # Food insecurity — Feeding America 2024
    "Food_Insecurity_Year",
    "Food_Insecurity_Rate",
    "Food_Insecure_Persons",
    "Child_Food_Insecurity_Rate",
    "Food_Insecure_Children",
    "SNAP_Threshold",
    "FI_At_or_Below_SNAP_Threshold_Pct",
    "FI_Above_SNAP_Threshold_Pct",
    "Cost_Per_Meal",
    "Annual_Food_Budget_Shortfall",

    # Geographic classifications
    "RUCC_2023",
    "Metro_Status",
    "Region",
    "Census_Division",

    # Health insurance — SAHIE 2024
    "Uninsured_Year",
    "Uninsured_Rate",
    "Uninsured_Persons",
]

master_final = master_final[final_column_order]

In [39]:
assert master_final.shape == (3144, 32)
assert master_final["FIPS"].nunique() == 3144
assert master_final["FIPS"].duplicated().sum() == 0

print("Schema reordered successfully.")
print("Shape:", master_final.shape)

Schema reordered successfully.
Shape: (3144, 32)


In [40]:
missing = master_final.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("COLUMNS CONTAINING MISSING VALUES")
print("-" * 40)

if missing.empty:
    print("No missing values.")
else:
    display(
        pd.DataFrame({
            "Missing_Count": missing,
            "Missing_Pct": (missing / len(master_final) * 100).round(2)
        })
    )

COLUMNS CONTAINING MISSING VALUES
----------------------------------------


,Missing_Count,Missing_Pct
Food_Insecurity_Rate,2,0.06
Food_Insecure_Persons,2,0.06
Annual_Food_Budget_Shortfall,2,0.06
FI_Above_SNAP_Threshold_Pct,2,0.06
FI_At_or_Below_SNAP_Threshold_Pct,2,0.06
Food_Insecure_Children,2,0.06
Child_Food_Insecurity_Rate,2,0.06
Uninsured_Rate,1,0.03
Poverty_All_Pct,1,0.03
Poverty_Index,1,0.03


In [41]:
# Identify county with missing Economic Stress Score

stress_missing = master_final.loc[
    master_final["Economic_Stress_Score"].isna(),
    [
        "FIPS",
        "State",
        "County",
        "Poverty_All_Pct",
        "Median_Household_Income",
        "Unemployment_Pct",
        "Severe_Burden_Pct",
        "Economic_Stress_Score",
        "Food_Insecurity_Rate",
        "Uninsured_Rate"
    ]
]

display(stress_missing)

,FIPS,State,County,Poverty_All_Pct,Median_Household_Income,Unemployment_Pct,Severe_Burden_Pct,Economic_Stress_Score,Food_Insecurity_Rate,Uninsured_Rate
550,15005,HI,Kalawao County,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Source-Level Missingness

Kalawao County, Hawaii (FIPS 15005) lacks several source measures required
for the Economic Stress Score and also lacks published overall food insecurity
and uninsured estimates in the integrated Feeding America and SAHIE data.

Loving County, Texas (FIPS 48301) lacks selected Feeding America food
insecurity estimates.

Both geographies are retained in the canonical 3,144-county master. Missing
source estimates remain null and are not imputed or interpreted as zero.

In [42]:
qa_fields = [
    "Poverty_All_Pct",
    "Median_Household_Income",
    "Unemployment_Pct",
    "Severe_Burden_Pct",
    "High_Burden_35Plus_Pct",
    "Economic_Stress_Score",
    "Food_Insecurity_Rate",
    "Child_Food_Insecurity_Rate",
    "SNAP_Threshold",
    "FI_At_or_Below_SNAP_Threshold_Pct",
    "FI_Above_SNAP_Threshold_Pct",
    "Cost_Per_Meal",
    "Annual_Food_Budget_Shortfall",
    "Uninsured_Rate"
]

master_final[qa_fields].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Poverty_All_Pct,3143.0,1.410800e+01,5.437000e+00,3.800,10.350,13.200,16.800,5.570000e+01
Median_Household_Income,3143.0,6.949763e+04,1.723342e+04,34802.000,58374.000,66757.000,77138.000,1.774570e+05
Unemployment_Pct,3143.0,4.181000e+00,1.310000e+00,0.700,3.400,4.000,4.700,1.890000e+01
Severe_Burden_Pct,3143.0,1.758900e+01,6.648000e+00,0.000,13.431,17.800,21.992,6.395400e+01
High_Burden_35Plus_Pct,3144.0,2.945800e+01,8.978000e+00,0.000,24.085,29.966,35.656,6.395300e+01
Economic_Stress_Score,3143.0,2.215500e+01,6.329000e+00,1.600,17.900,21.700,25.800,7.340000e+01
Food_Insecurity_Rate,3142.0,1.640000e-01,4.000000e-02,0.058,0.135,0.161,0.188,3.710000e-01
Child_Food_Insecurity_Rate,3142.0,2.250000e-01,5.500000e-02,0.066,0.187,0.220,0.260,4.450000e-01
SNAP_Threshold,3144.0,1.684000e+00,3.090000e-01,1.300,1.300,1.650,2.000,2.000000e+00
FI_At_or_Below_SNAP_Threshold_Pct,3142.0,5.660000e-01,1.530000e-01,0.028,0.460,0.562,0.671,1.000000e+00


In [43]:
snap_sum = (
    master_final["FI_At_or_Below_SNAP_Threshold_Pct"]
    + master_final["FI_Above_SNAP_Threshold_Pct"]
)

print("SNAP share sum range:")
print(snap_sum.min(), "to", snap_sum.max())

print(
    "Count not approximately equal to 1:",
    (~snap_sum.dropna().between(0.999, 1.001)).sum()
)

SNAP share sum range:
1.0 to 1.0
Count not approximately equal to 1: 0


In [44]:
vt_check = master_final[
    master_final["State"] == "VT"
][
    [
        "FIPS",
        "County",
        "Economic_Stress_Score",
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Food_Insecurity_Rate",
        "Child_Food_Insecurity_Rate",
        "Uninsured_Rate",
        "RUCC_2023",
        "Metro_Status",
        "Region",
        "Census_Division"
    ]
].sort_values("County")

print("Vermont county count:", len(vt_check))
display(vt_check)

Vermont county count: 14


,FIPS,County,Economic_Stress_Score,Poverty_All_Pct,Unemployment_Pct,Food_Insecurity_Rate,Child_Food_Insecurity_Rate,Uninsured_Rate,RUCC_2023,Metro_Status,Region,Census_Division
2808,50001,Addison County,16.9,8.0,2.5,0.117,0.158,4.9,6,Nonmetropolitan,Northeast,New England
2809,50003,Bennington County,25.7,9.9,3.1,0.144,0.204,6.0,6,Nonmetropolitan,Northeast,New England
2810,50005,Caledonia County,21.1,11.0,2.8,0.139,0.187,5.9,9,Nonmetropolitan,Northeast,New England
2811,50007,Chittenden County,19.5,8.9,2.2,0.119,0.138,4.3,3,Metropolitan,Northeast,New England
2812,50009,Essex County,20.4,13.8,3.6,0.163,0.228,6.6,9,Nonmetropolitan,Northeast,New England
2813,50011,Franklin County,18.5,8.8,2.7,0.128,0.181,4.5,3,Metropolitan,Northeast,New England
2814,50013,Grand Isle County,19.9,8.0,3.3,0.102,0.155,4.6,3,Metropolitan,Northeast,New England
2815,50015,Lamoille County,15.3,8.9,3.0,0.120,0.188,7.3,8,Nonmetropolitan,Northeast,New England
2816,50017,Orange County,18.3,9.8,2.9,0.125,0.186,5.3,9,Nonmetropolitan,Northeast,New England
2817,50019,Orleans County,26.1,11.9,4.3,0.144,0.207,6.1,9,Nonmetropolitan,Northeast,New England


## Economic Stress Index — Methodology Review

The original Economic Stress Score combines county-level poverty,
unemployment, and severe housing burden indicators.

Before publication of the County Explorer, the composite measure is
reviewed to determine whether its construction, normalization, and
weighting remain appropriate for a broader public-facing analytical tool.

Food insecurity and health insurance coverage are retained as independent
measures rather than incorporated into the index, allowing their
relationships with economic stress to be examined separately.

In [45]:
# Inspect Economic Stress component relationships

stress_components = [
    "Poverty_All_Pct",
    "Poverty_Index",
    "Unemployment_Pct",
    "Unemployment_Index",
    "Severe_Burden_Pct",
    "Housing_Index",
    "Economic_Stress_Score"
]

display(master_final[stress_components].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
Poverty_All_Pct,3143.0,14.108,5.437,3.8,10.350,13.200,16.800,55.700
Poverty_Index,3143.0,0.199,0.105,0.0,0.126,0.181,0.250,1.000
Unemployment_Pct,3143.0,4.181,1.310,0.7,3.400,4.000,4.700,18.900
Unemployment_Index,3143.0,0.191,0.072,0.0,0.148,0.181,0.220,1.000
Severe_Burden_Pct,3143.0,17.589,6.648,0.0,13.431,17.800,21.992,63.954
Housing_Index,3143.0,0.275,0.104,0.0,0.210,0.278,0.344,0.999
Economic_Stress_Score,3143.0,22.155,6.329,1.6,17.900,21.700,25.800,73.400


In [48]:
calculated_score = (
    master_final[
        [
            "Poverty_Index",
            "Unemployment_Index",
            "Housing_Index"
        ]
    ].mean(axis=1) * 100
)

difference = (
    master_final["Economic_Stress_Score"] - calculated_score
).abs()

print("Maximum difference:", difference.max())
print("Mean difference:", difference.mean())
print(
    "Rows matching within 0.1 points:",
    (difference <= 0.1).sum(),
    "of",
    difference.notna().sum()
)

Maximum difference: 0.03333333333334565
Mean difference: 0.02240958744299563
Rows matching within 0.1 points: 3143 of 3143


In [49]:
stress_corr = master_final[
    [
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Severe_Burden_Pct"
    ]
].corr()

display(stress_corr.round(3))

,Poverty_All_Pct,Unemployment_Pct,Severe_Burden_Pct
Poverty_All_Pct,1.000,0.400,0.042
Unemployment_Pct,0.400,1.000,0.143
Severe_Burden_Pct,0.042,0.143,1.000


In [50]:
# Examine potential outliers in the three Economic Stress components

for col in [
    "Poverty_All_Pct",
    "Unemployment_Pct",
    "Severe_Burden_Pct"
]:
    q1 = master_final[col].quantile(0.25)
    q3 = master_final[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = master_final[
        (master_final[col] < lower_bound) |
        (master_final[col] > upper_bound)
    ]

    print(f"\n{col}")
    print("-" * 40)
    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"IQR: {iqr:.2f}")
    print(f"Upper outlier threshold: {upper_bound:.2f}")
    print(f"Number of outlier counties: {len(outliers)}")
    print(f"Maximum observed value: {master_final[col].max():.2f}")


Poverty_All_Pct
----------------------------------------
Q1: 10.35
Q3: 16.80
IQR: 6.45
Upper outlier threshold: 26.48
Number of outlier counties: 90
Maximum observed value: 55.70

Unemployment_Pct
----------------------------------------
Q1: 3.40
Q3: 4.70
IQR: 1.30
Upper outlier threshold: 6.65
Number of outlier counties: 132
Maximum observed value: 18.90

Severe_Burden_Pct
----------------------------------------
Q1: 13.43
Q3: 21.99
IQR: 8.56
Upper outlier threshold: 34.83
Number of outlier counties: 40
Maximum observed value: 63.95


In [51]:
for col in [
    "Poverty_All_Pct",
    "Unemployment_Pct",
    "Severe_Burden_Pct"
]:
    print(f"\nTOP 10 — {col}")
    print("-" * 50)

    display(
        master_final[
            ["FIPS", "State", "County", col]
        ]
        .nlargest(10, col)
        .reset_index(drop=True)
    )


TOP 10 — Poverty_All_Pct
--------------------------------------------------


,FIPS,State,County,Poverty_All_Pct
0,28055,MS,Issaquena County,55.7
1,46137,SD,Ziebach County,45.8
2,22035,LA,East Carroll Parish,44.4
3,05077,AR,Lee County,41.3
4,08025,CO,Crowley County,39.0
5,54047,WV,McDowell County,38.4
6,46031,SD,Corson County,37.9
7,13037,GA,Calhoun County,36.8
8,01011,AL,Bullock County,36.7
9,21147,KY,McCreary County,36.4



TOP 10 — Unemployment_Pct
--------------------------------------------------


,FIPS,State,County,Unemployment_Pct
0,06025,CA,Imperial County,18.9
1,02158,AK,Kusilvak Census Area,17.4
2,04027,AZ,Yuma County,14.0
3,06011,CA,Colusa County,13.9
4,22035,LA,East Carroll Parish,13.0
5,35029,NM,Luna County,13.0
6,02050,AK,Bethel Census Area,12.0
7,26135,MI,Oscoda County,11.0
8,06107,CA,Tulare County,10.8
9,26097,MI,Mackinac County,10.8



TOP 10 — Severe_Burden_Pct
--------------------------------------------------


,FIPS,State,County,Severe_Burden_Pct
0,48243,TX,Jeff Davis County,63.9535
1,32009,NV,Esmeralda County,56.9767
2,22035,LA,East Carroll Parish,47.7092
3,37189,NC,Watauga County,42.2006
4,32011,NV,Eureka County,39.1304
5,13141,GA,Hancock County,39.0966
6,53075,WA,Whitman County,37.1440
7,51670,VA,Hopewell city,36.6046
8,28143,MS,Tunica County,35.9756
9,48155,TX,Foard County,35.7143


In [52]:
top_esi = (
    master_final[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Poverty_Index",
            "Unemployment_Pct",
            "Unemployment_Index",
            "Severe_Burden_Pct",
            "Housing_Index",
            "Economic_Stress_Score"
        ]
    ]
    .nlargest(20, "Economic_Stress_Score")
    .reset_index(drop=True)
)

display(top_esi)

,FIPS,State,County,Poverty_All_Pct,Poverty_Index,Unemployment_Pct,Unemployment_Index,Severe_Burden_Pct,Housing_Index,Economic_Stress_Score
0,22035,LA,East Carroll Parish,44.4,0.782,13.0,0.676,47.7092,0.745,73.4
1,06025,CA,Imperial County,18.8,0.289,18.9,1.000,25.6205,0.400,56.3
2,02158,AK,Kusilvak Census Area,34.9,0.599,17.4,0.918,9.3426,0.146,55.4
3,28055,MS,Issaquena County,55.7,1.000,9.3,0.473,3.6232,0.057,51.0
4,48427,TX,Starr County,35.4,0.609,9.5,0.484,24.5621,0.384,49.2
5,54047,WV,McDowell County,38.4,0.667,9.9,0.505,19.1381,0.299,49.0
6,32009,NV,Esmeralda County,14.8,0.212,7.3,0.363,56.9767,0.890,48.8
7,05077,AR,Lee County,41.3,0.723,6.2,0.302,27.5391,0.430,48.5
8,35029,NM,Luna County,23.9,0.387,13.0,0.676,20.2500,0.316,46.0
9,48243,TX,Jeff Davis County,12.8,0.173,4.4,0.203,63.9535,0.999,45.8


In [53]:
component_indexes = master_final[
    [
        "Poverty_Index",
        "Unemployment_Index",
        "Housing_Index"
    ]
]

master_final["Primary_Stress_Driver"] = (
    component_indexes.idxmax(axis=1)
    .replace({
        "Poverty_Index": "Poverty",
        "Unemployment_Index": "Unemployment",
        "Housing_Index": "Severe Housing Burden"
    })
)

print(
    master_final["Primary_Stress_Driver"]
    .value_counts()
)

print("\nPercent of counties:")
print(
    master_final["Primary_Stress_Driver"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

Primary_Stress_Driver
Severe Housing Burden    2091
Poverty                   723
Unemployment              329
Name: count, dtype: int64

Percent of counties:
Primary_Stress_Driver
Severe Housing Burden    66.5
Poverty                  23.0
Unemployment             10.5
Name: proportion, dtype: float64


/var/folders/g0/c688s8gd357cfm74km_mjqh00000gn/T/ipykernel_56646/2263706458.py:10: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  component_indexes.idxmax(axis=1)


In [54]:
# Sensitivity test: z-score standardized components

z_components = master_final[
    [
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Severe_Burden_Pct"
    ]
].copy()

z_components["Poverty_Z"] = (
    z_components["Poverty_All_Pct"]
    - z_components["Poverty_All_Pct"].mean()
) / z_components["Poverty_All_Pct"].std()

z_components["Unemployment_Z"] = (
    z_components["Unemployment_Pct"]
    - z_components["Unemployment_Pct"].mean()
) / z_components["Unemployment_Pct"].std()

z_components["Housing_Z"] = (
    z_components["Severe_Burden_Pct"]
    - z_components["Severe_Burden_Pct"].mean()
) / z_components["Severe_Burden_Pct"].std()

z_components["Primary_Z_Driver"] = (
    z_components[
        ["Poverty_Z", "Unemployment_Z", "Housing_Z"]
    ]
    .idxmax(axis=1)
    .replace({
        "Poverty_Z": "Poverty",
        "Unemployment_Z": "Unemployment",
        "Housing_Z": "Severe Housing Burden"
    })
)

print(z_components["Primary_Z_Driver"].value_counts())

print("\nPercent of counties:")
print(
    z_components["Primary_Z_Driver"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

Primary_Z_Driver
Severe Housing Burden    1327
Poverty                  1014
Unemployment              802
Name: count, dtype: int64

Percent of counties:
Primary_Z_Driver
Severe Housing Burden    42.2
Poverty                  32.3
Unemployment             25.5
Name: proportion, dtype: float64


/var/folders/g0/c688s8gd357cfm74km_mjqh00000gn/T/ipykernel_56646/2971676461.py:30: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmax(axis=1)


In [55]:
master_final.drop(
    columns=["Primary_Stress_Driver"],
    inplace=True,
    errors="ignore"
)

print(master_final.shape)

(3144, 32)


### Sensitivity Analysis

A sensitivity test compared the existing min-max normalization with
z-score standardization for the three Economic Stress components.

Under min-max normalization, severe housing burden was the largest
normalized component in 66.5% of counties, compared with 23.0% for
poverty and 10.5% for unemployment.

After z-score standardization, the distribution was more balanced:
42.2% severe housing burden, 32.3% poverty, and 25.5% unemployment.

This indicates that component dominance in the original index is
partly sensitive to the normalization method. The result does not by
itself determine which method is preferable; normalization methodology
should be evaluated before finalizing the public-facing index.

In [46]:
# Export final canonical county master

excel_output = OUTPUT_DIR / "County_Explorer_Master_2026.xlsx"
csv_output = OUTPUT_DIR / "County_Explorer_Master_2026.csv"

master_final.to_excel(
    excel_output,
    index=False
)

master_final.to_csv(
    csv_output,
    index=False
)

print("Export complete.")
print("Excel:", excel_output)
print("CSV:", csv_output)

Export complete.
Excel: /Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/outputs/County_Explorer_Master_2026.xlsx
CSV: /Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/outputs/County_Explorer_Master_2026.csv


## ESI-C — Percentile-Rank Sensitivity Analysis

This alternative Economic Stress Index uses percentile ranks rather than
min-max normalization.

County poverty, unemployment, and severe housing burden are each ranked
relative to all U.S. counties with available data. The three component
percentile ranks are equally weighted and averaged on a 0–100 scale.

Higher values indicate greater relative economic stress.

In [56]:
# Build percentile-rank ESI candidate

esi_c = master_final[
    [
        "FIPS",
        "State",
        "County",
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Severe_Burden_Pct",
        "Economic_Stress_Score"
    ]
].copy()

esi_c["Poverty_Percentile"] = (
    esi_c["Poverty_All_Pct"]
    .rank(method="average", pct=True)
)

esi_c["Unemployment_Percentile"] = (
    esi_c["Unemployment_Pct"]
    .rank(method="average", pct=True)
)

esi_c["Housing_Percentile"] = (
    esi_c["Severe_Burden_Pct"]
    .rank(method="average", pct=True)
)

esi_c["ESI_C_Percentile"] = (
    esi_c[
        [
            "Poverty_Percentile",
            "Unemployment_Percentile",
            "Housing_Percentile"
        ]
    ]
    .mean(axis=1)
    * 100
)

esi_c.head()

,FIPS,State,County,Poverty_All_Pct,Unemployment_Pct,Severe_Burden_Pct,Economic_Stress_Score,Poverty_Percentile,Unemployment_Percentile,Housing_Percentile,ESI_C_Percentile
0,01001,AL,Autauga County,12.9,2.4,15.0770,16.8,0.477410,0.033248,0.336621,28.242656
1,01003,AL,Baldwin County,10.5,2.5,18.3905,17.2,0.263920,0.042953,0.533567,28.014636
2,01005,AL,Barbour County,28.1,3.6,14.3970,28.4,0.976137,0.345052,0.300350,54.051331
3,01007,AL,Bibb County,21.6,2.7,22.3380,26.7,0.913140,0.065224,0.768374,58.224626
4,01009,AL,Blount County,11.7,2.4,17.9396,17.5,0.375756,0.033248,0.509068,30.602397


In [57]:
print("Rows:", len(esi_c))
print("Missing ESI-C:", esi_c["ESI_C_Percentile"].isna().sum())

display(
    esi_c[
        [
            "Poverty_Percentile",
            "Unemployment_Percentile",
            "Housing_Percentile",
            "ESI_C_Percentile"
        ]
    ]
    .describe()
    .T
    .round(3)
)

Rows: 3144
Missing ESI-C: 1


,count,mean,std,min,25%,50%,75%,max
Poverty_Percentile,3143.0,0.500,0.289,0.000,0.251,0.499,0.748,1.00
Unemployment_Percentile,3143.0,0.500,0.289,0.000,0.263,0.504,0.741,1.00
Housing_Percentile,3143.0,0.500,0.289,0.005,0.250,0.500,0.750,1.00
ESI_C_Percentile,3143.0,50.016,19.761,1.008,35.136,50.753,64.728,99.91


In [58]:
top_esi_c = (
    esi_c[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Poverty_Percentile",
            "Unemployment_Pct",
            "Unemployment_Percentile",
            "Severe_Burden_Pct",
            "Housing_Percentile",
            "Economic_Stress_Score",
            "ESI_C_Percentile"
        ]
    ]
    .nlargest(20, "ESI_C_Percentile")
    .reset_index(drop=True)
)

display(top_esi_c)

,FIPS,State,County,Poverty_All_Pct,Poverty_Percentile,Unemployment_Pct,Unemployment_Percentile,Severe_Burden_Pct,Housing_Percentile,Economic_Stress_Score,ESI_C_Percentile
0,22035,LA,East Carroll Parish,44.4,0.999364,13.0,0.998568,47.7092,0.999364,73.4,99.909853
1,36005,NY,Bronx County,28.7,0.978683,7.0,0.969615,32.9091,0.991091,44.7,97.979637
2,45089,SC,Williamsburg County,25.5,0.961979,7.7,0.982342,27.6175,0.945912,41.2,96.341075
3,05077,AR,Lee County,41.3,0.999045,6.2,0.940662,27.5391,0.943684,48.5,96.113055
4,05107,AR,Phillips County,35.8,0.996341,7.1,0.972001,25.3779,0.893096,45.5,95.381271
5,48427,TX,Starr County,35.4,0.995546,9.5,0.994114,24.5621,0.867642,49.2,95.243398
6,39009,OH,Athens County,25.3,0.960070,5.6,0.895959,34.1232,0.995227,40.5,95.041892
7,21231,KY,Wayne County,28.8,0.979319,5.9,0.920935,27.2566,0.936685,39.8,94.564641
8,54015,WV,Clay County,20.1,0.876233,7.7,0.982342,29.0530,0.967547,38.4,94.204051
9,22067,LA,Morehouse Parish,28.5,0.978046,6.2,0.940662,25.8806,0.902959,39.4,94.055573


In [59]:
esi_c = esi_c.rename(
    columns={
        "ESI_C_Percentile": "Mean_Component_Percentile"
    }
)

esi_c["ESI_C_Final"] = (
    esi_c["Mean_Component_Percentile"]
    .rank(method="average", pct=True)
    * 100
)

display(
    esi_c[
        [
            "FIPS",
            "State",
            "County",
            "Mean_Component_Percentile",
            "ESI_C_Final"
        ]
    ]
    .nlargest(20, "ESI_C_Final")
    .reset_index(drop=True)
)

,FIPS,State,County,Mean_Component_Percentile,ESI_C_Final
0,22035,LA,East Carroll Parish,99.909853,100.000000
1,36005,NY,Bronx County,97.979637,99.968183
2,45089,SC,Williamsburg County,96.341075,99.936367
3,05077,AR,Lee County,96.113055,99.904550
4,05107,AR,Phillips County,95.381271,99.872733
5,48427,TX,Starr County,95.243398,99.840916
6,39009,OH,Athens County,95.041892,99.809100
7,21231,KY,Wayne County,94.564641,99.777283
8,54015,WV,Clay County,94.204051,99.745466
9,22067,LA,Morehouse Parish,94.055573,99.713649


In [60]:
# Compare original min-max ESI with percentile-based ESI-C

comparison = esi_c[
    [
        "FIPS",
        "State",
        "County",
        "Economic_Stress_Score",
        "Mean_Component_Percentile",
        "ESI_C_Final"
    ]
].copy()

# Rank 1 = highest economic stress
comparison["MinMax_Rank"] = (
    comparison["Economic_Stress_Score"]
    .rank(method="min", ascending=False)
)

comparison["Percentile_Rank"] = (
    comparison["ESI_C_Final"]
    .rank(method="min", ascending=False)
)

comparison["Rank_Change"] = (
    comparison["MinMax_Rank"]
    - comparison["Percentile_Rank"]
)

comparison["Absolute_Rank_Change"] = (
    comparison["Rank_Change"].abs()
)

display(
    comparison
    .nlargest(20, "Absolute_Rank_Change")
    .reset_index(drop=True)
)

,FIPS,State,County,Economic_Stress_Score,Mean_Component_Percentile,ESI_C_Final,MinMax_Rank,Percentile_Rank,Rank_Change,Absolute_Rank_Change
0,13309,GA,Wheeler County,31.1,45.147948,40.948139,241.0,1857.0,-1616.0,1616.0
1,13037,GA,Calhoun County,33.3,49.310637,47.216036,153.0,1660.0,-1507.0,1507.0
2,13167,GA,Johnson County,30.2,48.377346,45.625199,288.0,1710.0,-1422.0,1422.0
3,46007,SD,Bennett County,26.9,42.766995,37.448298,619.0,1967.0,-1348.0,1348.0
4,28135,MS,Tallahatchie County,28.9,48.897020,46.516067,391.0,1682.0,-1291.0,1291.0
5,13315,GA,Wilcox County,26.7,47.025135,43.620745,637.0,1773.0,-1136.0,1136.0
6,38085,ND,Sioux County,30.2,53.669530,55.345212,288.0,1404.0,-1116.0,1116.0
7,13003,GA,Atkinson County,28.2,51.511295,51.447661,449.0,1527.0,-1078.0,1078.0
8,46047,SD,Fall River County,26.5,47.852370,44.782055,666.0,1736.0,-1070.0,1070.0
9,13165,GA,Jenkins County,29.4,53.409693,54.899777,350.0,1418.0,-1068.0,1068.0


In [61]:
spearman = comparison[
    ["Economic_Stress_Score", "ESI_C_Final"]
].corr(method="spearman").iloc[0, 1]

pearson = comparison[
    ["Economic_Stress_Score", "Mean_Component_Percentile"]
].corr(method="pearson").iloc[0, 1]

print(f"Spearman rank correlation: {spearman:.3f}")
print(f"Pearson score correlation: {pearson:.3f}")

Spearman rank correlation: 0.970
Pearson score correlation: 0.933


Now I want to understand the rebels.

In [62]:
case_fips = [
    "13309",  # Wheeler, GA
    "46137",  # Ziebach, SD
    "50003",  # Bennington, VT
    "35011",  # DeBaca, NM
    "17151",  # Pope, IL
    "36005",  # Bronx, NY
    "22035"   # East Carroll, LA
]

case_study = (
    master_final[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Poverty_Index",
            "Unemployment_Pct",
            "Unemployment_Index",
            "Severe_Burden_Pct",
            "Housing_Index",
            "Economic_Stress_Score"
        ]
    ]
    .merge(
        esi_c[
            [
                "FIPS",
                "Poverty_Percentile",
                "Unemployment_Percentile",
                "Housing_Percentile",
                "Mean_Component_Percentile",
                "ESI_C_Final"
            ]
        ],
        on="FIPS",
        how="left"
    )
)

case_study = case_study[
    case_study["FIPS"].isin(case_fips)
].copy()

case_study["MinMax_Rank"] = case_study["FIPS"].map(
    comparison.set_index("FIPS")["MinMax_Rank"]
)

case_study["Percentile_Rank"] = case_study["FIPS"].map(
    comparison.set_index("FIPS")["Percentile_Rank"]
)

display(case_study.round(3))

,FIPS,State,County,Poverty_All_Pct,Poverty_Index,Unemployment_Pct,Unemployment_Index,Severe_Burden_Pct,Housing_Index,Economic_Stress_Score,Poverty_Percentile,Unemployment_Percentile,Housing_Percentile,Mean_Component_Percentile,ESI_C_Final,MinMax_Rank,Percentile_Rank
541,13309,GA,Wheeler County,35.0,0.601,2.9,0.121,13.486,0.211,31.1,0.995,0.107,0.252,45.148,40.948,241.0,1857.0
672,17151,IL,Pope County,16.5,0.245,5.3,0.253,2.105,0.033,17.7,0.733,0.861,0.016,53.648,55.266,2381.0,1407.0
1132,22035,LA,East Carroll Parish,44.4,0.782,13.0,0.676,47.709,0.745,73.4,0.999,0.999,0.999,99.910,100.000,1.0,1.0
1803,35011,NM,DeBaca County,16.9,0.252,4.4,0.203,0.000,0.000,15.2,0.755,0.653,0.005,47.089,43.653,2762.0,1772.0
1832,36005,NY,Bronx County,28.7,0.480,7.0,0.346,32.909,0.514,44.7,0.979,0.970,0.991,97.980,99.968,14.0,2.0
2429,46137,SD,Ziebach County,45.8,0.809,2.9,0.121,20.659,0.323,41.8,1.000,0.107,0.677,59.455,65.956,22.0,1070.0
2809,50003,VT,Bennington County,9.9,0.118,3.1,0.132,33.436,0.522,25.7,0.219,0.160,0.993,45.752,41.998,801.0,1824.0


## ESI-B — Z-Score Standardization Sensitivity Analysis

This alternative Economic Stress Index standardizes poverty,
unemployment, and severe housing burden using z-scores.

Each component represents the number of standard deviations a county
falls above or below the national county mean for that indicator.
The three standardized components are equally weighted and averaged
to create a composite standardized Economic Stress measure.

This candidate is used to evaluate whether standardization preserves
meaningful differences in severity while reducing sensitivity to the
observed minimum and maximum values.

In [63]:
# Build ESI-B using z-score standardized components

esi_b = master_final[
    [
        "FIPS",
        "State",
        "County",
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Severe_Burden_Pct",
        "Economic_Stress_Score"
    ]
].copy()

esi_b["Poverty_Z"] = (
    esi_b["Poverty_All_Pct"] - esi_b["Poverty_All_Pct"].mean()
) / esi_b["Poverty_All_Pct"].std()

esi_b["Unemployment_Z"] = (
    esi_b["Unemployment_Pct"] - esi_b["Unemployment_Pct"].mean()
) / esi_b["Unemployment_Pct"].std()

esi_b["Housing_Z"] = (
    esi_b["Severe_Burden_Pct"] - esi_b["Severe_Burden_Pct"].mean()
) / esi_b["Severe_Burden_Pct"].std()

esi_b["ESI_B_Z_Composite"] = (
    esi_b[
        [
            "Poverty_Z",
            "Unemployment_Z",
            "Housing_Z"
        ]
    ].mean(axis=1)
)

In [64]:
display(
    esi_b[
        [
            "Poverty_Z",
            "Unemployment_Z",
            "Housing_Z",
            "ESI_B_Z_Composite"
        ]
    ]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
Poverty_Z,3143.0,-0.0,1.000,-1.896,-0.691,-0.167,0.495,7.649
Unemployment_Z,3143.0,0.0,1.000,-2.657,-0.596,-0.138,0.397,11.236
Housing_Z,3143.0,-0.0,1.000,-2.646,-0.626,0.032,0.662,6.975
ESI_B_Z_Composite,3143.0,-0.0,0.681,-2.246,-0.455,-0.042,0.382,5.611


In [65]:
print("Rows:", len(esi_b))
print(
    "Complete ESI-B:",
    esi_b["ESI_B_Z_Composite"].notna().sum()
)
print(
    "Missing ESI-B:",
    esi_b["ESI_B_Z_Composite"].isna().sum()
)

Rows: 3144
Complete ESI-B: 3143
Missing ESI-B: 1


In [66]:
top_esi_b = (
    esi_b[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Poverty_Z",
            "Unemployment_Pct",
            "Unemployment_Z",
            "Severe_Burden_Pct",
            "Housing_Z",
            "ESI_B_Z_Composite"
        ]
    ]
    .nlargest(20, "ESI_B_Z_Composite")
    .reset_index(drop=True)
)

display(top_esi_b.round(3))

,FIPS,State,County,Poverty_All_Pct,Poverty_Z,Unemployment_Pct,Unemployment_Z,Severe_Burden_Pct,Housing_Z,ESI_B_Z_Composite
0,22035,LA,East Carroll Parish,44.4,5.571,13.0,6.733,47.709,4.531,5.611
1,06025,CA,Imperial County,18.8,0.863,18.9,11.236,25.620,1.208,4.436
2,02158,AK,Kusilvak Census Area,34.9,3.824,17.4,10.091,9.343,-1.241,4.225
3,28055,MS,Issaquena County,55.7,7.649,9.3,3.908,3.623,-2.101,3.152
4,54047,WV,McDowell County,38.4,4.467,9.9,4.366,19.138,0.233,3.022
5,48427,TX,Starr County,35.4,3.916,9.5,4.061,24.562,1.049,3.008
6,35029,NM,Luna County,23.9,1.801,13.0,6.733,20.250,0.400,2.978
7,32009,NV,Esmeralda County,14.8,0.127,7.3,2.381,56.977,5.925,2.811
8,04027,AZ,Yuma County,16.6,0.458,14.0,7.496,18.826,0.186,2.713
9,05077,AR,Lee County,41.3,5.001,6.2,1.542,27.539,1.497,2.680


# Indicator Selection Question: Unemployment

The sensitivity analysis raised a conceptual concern about unemployment as a component of the Economic Stress Index. High unemployment clearly represents economic stress; however, low unemployment does not necessarily represent economic stability. The unemployment rate does not capture low wages, insufficient hours, multiple-job holding, lack of employment benefits, or people outside the labor force. Before finalizing the ESI, the labor-market component should therefore be reevaluated to determine whether unemployment should be retained, supplemented, replaced, or excluded.

## Production Implementation — Economic Stress Index — County (ESI-C)

The finalized ESI-C specification was developed in
`02_Labor_Market_Indicator_Research.ipynb`.

That investigation evaluated the original unemployment-based composite,
alternative labor-market indicators, ACS sampling uncertainty, component
relationships, and weighting sensitivity.

The production specification uses:

- Poverty percentile
- Prime-age did-not-work percentile
- Severe housing burden percentile

Each component receives equal weight.

The original `Economic_Stress_Score` is retained as the historical
class-project measure. ESI-C is added as a separate revised index rather
than replacing the original field.

In [67]:
# Production pull — ACS 2024 5-Year Detailed Table B23027
# Prime-age (25–54) population and did-not-work counts

B23027_URL = "https://api.census.gov/data/2024/acs/acs5"

b23027_params = {
    "get": ",".join([
        "NAME",
        "B23027_012E",  # Age 25–44 total population
        "B23027_016E",  # Age 25–44 did not work
        "B23027_017E",  # Age 45–54 total population
        "B23027_021E"   # Age 45–54 did not work
    ]),
    "for": "county:*",
    "in": "state:*",
    "key": CENSUS_API_KEY
}

b23027_response = requests.get(
    B23027_URL,
    params=b23027_params,
    timeout=30
)

print("Status code:", b23027_response.status_code)

Status code: 200


In [68]:
# Convert B23027 response to DataFrame

b23027_data = b23027_response.json()

labor_esi = pd.DataFrame(
    b23027_data[1:],
    columns=b23027_data[0]
)

labor_esi["FIPS"] = (
    labor_esi["state"].str.zfill(2)
    + labor_esi["county"].str.zfill(3)
)

# Exclude Puerto Rico to match the canonical 3,144-county universe
labor_esi = labor_esi[
    labor_esi["state"] != "72"
].copy()

count_cols = [
    "B23027_012E",
    "B23027_016E",
    "B23027_017E",
    "B23027_021E"
]

labor_esi[count_cols] = (
    labor_esi[count_cols]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows:", len(labor_esi))
print("Unique FIPS:", labor_esi["FIPS"].nunique())
print("Duplicate FIPS:", labor_esi["FIPS"].duplicated().sum())

Rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [69]:
# Construct prime-age 25–54 population and non-work measure

labor_esi["Prime_Age_Pop"] = (
    labor_esi["B23027_012E"]
    + labor_esi["B23027_017E"]
)

labor_esi["Prime_Age_Did_Not_Work"] = (
    labor_esi["B23027_016E"]
    + labor_esi["B23027_021E"]
)

labor_esi["Prime_Age_Did_Not_Work_Pct"] = (
    labor_esi["Prime_Age_Did_Not_Work"]
    / labor_esi["Prime_Age_Pop"]
    * 100
)

In [70]:
print("Valid prime-age non-work rates:",
      labor_esi["Prime_Age_Did_Not_Work_Pct"].notna().sum())

print("Missing prime-age non-work rates:",
      labor_esi["Prime_Age_Did_Not_Work_Pct"].isna().sum())

display(
    labor_esi.loc[
        labor_esi["Prime_Age_Did_Not_Work_Pct"].isna(),
        [
            "FIPS",
            "NAME",
            "Prime_Age_Pop",
            "Prime_Age_Did_Not_Work",
            "Prime_Age_Did_Not_Work_Pct"
        ]
    ]
)

Valid prime-age non-work rates: 3143
Missing prime-age non-work rates: 1


,FIPS,NAME,Prime_Age_Pop,Prime_Age_Did_Not_Work,Prime_Age_Did_Not_Work_Pct
2675,48301,"Loving County, Texas",0,0,NaN


In [71]:
# Merge finalized prime-age non-work measure into canonical master

master_final = master_final.merge(
    labor_esi[
        [
            "FIPS",
            "Prime_Age_Did_Not_Work_Pct"
        ]
    ],
    on="FIPS",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(master_final))
print("Columns:", master_final.shape[1])
print("Unique FIPS:", master_final["FIPS"].nunique())
print("Duplicate FIPS:", master_final["FIPS"].duplicated().sum())

Rows: 3144
Columns: 33
Unique FIPS: 3144
Duplicate FIPS: 0


In [72]:
# Create finalized ESI-C component percentiles

master_final["ESI_Poverty_Percentile"] = (
    master_final["Poverty_All_Pct"]
    .rank(method="average", pct=True)
)

master_final["ESI_NonWork_Percentile"] = (
    master_final["Prime_Age_Did_Not_Work_Pct"]
    .rank(method="average", pct=True)
)

master_final["ESI_Housing_Percentile"] = (
    master_final["Severe_Burden_Pct"]
    .rank(method="average", pct=True)
)

In [73]:
# Final Economic Stress Index — County (ESI-C)

master_final["ESI_C"] = (
    master_final[
        [
            "ESI_Poverty_Percentile",
            "ESI_NonWork_Percentile",
            "ESI_Housing_Percentile"
        ]
    ]
    .mean(axis=1, skipna=False)
    * 100
)

In [74]:
print("Valid ESI-C values:", master_final["ESI_C"].notna().sum())
print("Missing ESI-C values:", master_final["ESI_C"].isna().sum())

display(
    master_final.loc[
        master_final["ESI_C"].isna(),
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "Severe_Burden_Pct",
            "ESI_C"
        ]
    ]
)

Valid ESI-C values: 3142
Missing ESI-C values: 2


,FIPS,State,County,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,ESI_C
550,15005,HI,Kalawao County,NaN,12.244898,NaN,NaN
2675,48301,TX,Loving County,6.3,NaN,0.0,NaN


In [75]:
print("Shape:", master_final.shape)
print("Unique FIPS:", master_final["FIPS"].nunique())
print("Duplicate FIPS:", master_final["FIPS"].duplicated().sum())

display(
    master_final[
        [
            "ESI_Poverty_Percentile",
            "ESI_NonWork_Percentile",
            "ESI_Housing_Percentile",
            "ESI_C"
        ]
    ]
    .describe()
    .T
    .round(3)
)

Shape: (3144, 37)
Unique FIPS: 3144
Duplicate FIPS: 0


,count,mean,std,min,25%,50%,75%,max
ESI_Poverty_Percentile,3143.0,0.500,0.289,0.000,0.251,0.499,0.748,1.000
ESI_NonWork_Percentile,3143.0,0.500,0.289,0.000,0.250,0.500,0.750,1.000
ESI_Housing_Percentile,3143.0,0.500,0.289,0.005,0.250,0.500,0.750,1.000
ESI_C,3142.0,50.029,20.318,2.381,33.873,50.376,65.780,99.809


In [76]:
final_column_order = [
    # Geography / identifiers
    "FIPS",
    "State_FIPS",
    "County_FIPS",
    "State",
    "County",

    # Economic conditions
    "Poverty_All_Pct",
    "Poverty_Index",
    "Median_Household_Income",

    "Unemployment_Pct",
    "Unemployment_Index",

    "Prime_Age_Did_Not_Work_Pct",

    "Severe_Burden_Pct",
    "High_Burden_35Plus_Pct",
    "Housing_Index",
    "Renter_Pop",

    # Historical class-project composite
    "Economic_Stress_Score",

    # Revised Economic Stress Index — County
    "ESI_Poverty_Percentile",
    "ESI_NonWork_Percentile",
    "ESI_Housing_Percentile",
    "ESI_C",

    # Food insecurity — Feeding America 2024
    "Food_Insecurity_Year",
    "Food_Insecurity_Rate",
    "Food_Insecure_Persons",
    "Child_Food_Insecurity_Rate",
    "Food_Insecure_Children",
    "SNAP_Threshold",
    "FI_At_or_Below_SNAP_Threshold_Pct",
    "FI_Above_SNAP_Threshold_Pct",
    "Cost_Per_Meal",
    "Annual_Food_Budget_Shortfall",

    # Geographic classifications
    "RUCC_2023",
    "Metro_Status",
    "Region",
    "Census_Division",

    # Health insurance — SAHIE 2024
    "Uninsured_Year",
    "Uninsured_Rate",
    "Uninsured_Persons",
]

master_final = master_final[final_column_order]

print("Schema reordered.")
print("Shape:", master_final.shape)

Schema reordered.
Shape: (3144, 37)


In [77]:
assert master_final.shape == (3144, 37)
assert master_final["FIPS"].nunique() == 3144
assert master_final["FIPS"].duplicated().sum() == 0

print("Final schema validation passed.")

Final schema validation passed.


### Production ESI-C Integration

The finalized Economic Stress Index — County (ESI-C) has been added to the
canonical master dataset following the methodology developed and tested in
`02_Labor_Market_Indicator_Research.ipynb`.

The historical `Economic_Stress_Score` is retained for continuity with the
original class project.

ESI-C uses national county percentile ranks for:

- Poverty
- Prime-age did-not-work rate
- Severe housing burden

The three components are equally weighted.

ESI-C is unavailable for Kalawao County, Hawaii, because required poverty and
housing inputs are unavailable, and for Loving County, Texas, because the ACS
prime-age denominator used to derive the non-work rate is zero.

In [79]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Project directory:")
print(PROJECT_DIR)

Project directory:
/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/notebooks


In [80]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent

print("Notebook directory:")
print(NOTEBOOK_DIR)

print("\nProject directory:")
print(PROJECT_DIR)

Notebook directory:
/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/notebooks

Project directory:
/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer


In [81]:
# Export canonical County Explorer master with finalized ESI-C

output_dir = PROJECT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    output_dir
    / "Economic_Stress_US_Counties_Master_ESI_C.xlsx"
)

master_final.to_excel(
    output_path,
    index=False
)

print("Export complete:")
print(output_path)

Export complete:
/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer/data/processed/Economic_Stress_US_Counties_Master_ESI_C.xlsx


In [82]:
# Validate exported workbook

export_check = pd.read_excel(
    output_path,
    dtype={"FIPS": str}
)

print("Exported shape:", export_check.shape)
print("Unique FIPS:", export_check["FIPS"].nunique())
print("Duplicate FIPS:", export_check["FIPS"].duplicated().sum())
print("Valid ESI-C:", export_check["ESI_C"].notna().sum())
print("Missing ESI-C:", export_check["ESI_C"].isna().sum())

assert export_check.shape == (3144, 37)
assert export_check["FIPS"].nunique() == 3144
assert export_check["FIPS"].duplicated().sum() == 0
assert export_check["ESI_C"].notna().sum() == 3142
assert export_check["ESI_C"].isna().sum() == 2

print("\nCanonical ESI-C master export validated successfully.")

Exported shape: (3144, 37)
Unique FIPS: 3144
Duplicate FIPS: 0
Valid ESI-C: 3142
Missing ESI-C: 2

Canonical ESI-C master export validated successfully.


## Production Export Complete

The canonical 37-field County Explorer master containing the finalized
Economic Stress Index — County (ESI-C) was successfully exported and
validated after read-back.

Final structure:

- 3,144 U.S. counties and county equivalents
- 37 fields
- 3,144 unique county FIPS
- 0 duplicate county records
- 3,142 valid ESI-C values
- 2 documented unavailable ESI-C values

The ESI-C methodology and supporting sensitivity analysis are documented
in `02_Labor_Market_Indicator_Research.ipynb`.

## Final Export

The canonical County Explorer master dataset was successfully exported
following structural, geographic, missingness, numeric, categorical,
county-level, and ESI-C quality assurance.

### Final Dataset

- 3,144 U.S. counties and county equivalents
- 37 fields
- 3,144 unique 5-digit county FIPS
- No duplicate county records
- No counties removed because of unavailable source estimates
- Source-level missing values retained as null
- 3,142 counties with complete ESI-C values
- 2 counties with documented unavailable ESI-C values

### Economic Stress Measures

The dataset retains the original `Economic_Stress_Score` developed for the
class project and adds the revised **Economic Stress Index — County (ESI-C)**.

ESI-C combines equally weighted national county percentile ranks for:

- Poverty
- Prime-age did-not-work rate
- Severe housing burden

The ESI-C methodology, labor-indicator investigation, ACS uncertainty audit,
component-balance analysis, and weighting-sensitivity analysis are documented
in `02_Labor_Market_Indicator_Research.ipynb`.

### Integrated Sources

- Economic Stress county master
- U.S. Census Bureau ACS 5-Year estimates — poverty, housing burden,
  and prime-age work status
- Feeding America Map the Meal Gap 2026 — 2024 estimates
- U.S. Census Bureau SAHIE — 2024 estimates
- USDA ERS Rural-Urban Continuum Codes — 2023
- U.S. Census Region and Division classifications

The exported dataset is intended to serve as the canonical analytical
source for the **U.S. County Economic Stress & Food Insecurity Explorer**.